In [1]:
import os, sys

# Make the notebook runnable both interactively (from the repo root) and via
# `jupyter nbconvert --execute` (kernel cwd = experiments/).
if not os.path.isdir(os.path.join(os.getcwd(), 'tskan')):
    _root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(_root)
    sys.path.insert(0, _root)


# Interpretable baselines (paper, Table II)

Trains the interpretable baselines of the paper on the expert-crafted feature
sets (paper, Section V-B): average bitrate, initial delay (first chunk's
rebuffering time), number of stallings, total stalling duration and number of
resolution switches. All models are fit on the train split and evaluated on
the held-out test split.

Set `DATASET` below to run the other dataset.

Run from the repository root: `python -m jupyter notebook experiments/04_baselines_interpretable.ipynb`


In [2]:
# Configuration: pick the dataset
DATASET = 'netflix2'      # or 'waterloo3'


In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error

FILES = {
    'netflix2':  f'data/netflix2/compnetflixpluspadded_{{split}}.ts',
    'waterloo3': f'data/waterloo3/compwaterloo_{{split}}.ts',
}[DATASET]

Xs, ys = {}, {}
for split in ['TRAIN', 'VALID', 'TEST']:
    df = pd.read_csv(FILES.format(split=split))
    df.columns = df.columns.str.strip()
    Xs[split] = df.drop('mos_zscore', axis=1).values
    ys[split] = df['mos_zscore'].values

print('features:', Xs['TRAIN'].shape[1], list(df.columns))
print('train', Xs['TRAIN'].shape, '| valid', Xs['VALID'].shape, '| test', Xs['TEST'].shape)


features: 5 ['avg_bitrate', 'initial_delay', 'stallings', 'stalling_duration', 'switches', 'mos_zscore']
train (282, 5) | valid (54, 5) | test (56, 5)


In [4]:
# Scaling (fitted on train only), as in the paper
scaler = RobustScaler()
X_train_sc = scaler.fit_transform(Xs['TRAIN'])
X_valid_sc = scaler.transform(Xs['VALID'])
X_test_sc  = scaler.transform(Xs['TEST'])


In [5]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Lasso
from interpret.glassbox import ExplainableBoostingRegressor
from dnamite.models import DNAMiteRegressor

models = {
    'Decision Tree': (DecisionTreeRegressor(max_depth=5, random_state=0), None),
    'Linear Regression': (LinearRegression(), None),
    'LASSO': (Lasso(alpha=0.01), None),
    'EBM': (ExplainableBoostingRegressor(random_state=42), None),
    'NAM': (DNAMiteRegressor(random_state=42), None),
}
feature_names = [c for c in df.columns if c != 'mos_zscore']
X_train_df = pd.DataFrame(X_train_sc, columns=feature_names)
X_test_df = pd.DataFrame(X_test_sc, columns=feature_names)


In [6]:
results = pd.DataFrame(columns=['model', 'test rmse', 'params'])
for name, (model, _) in models.items():
    if name == 'NAM':
        model.fit(X_train_df, ys['TRAIN'])
        y_pred = model.predict(X_test_df)
    else:
        model.fit(X_train_sc, ys['TRAIN'])
        y_pred = model.predict(X_test_sc)
    rmse = float(np.sqrt(mean_squared_error(ys['TEST'], y_pred)))
    results.loc[len(results)] = [name, rmse, '']
    print(f'{name} test RMSE: {rmse:.4f}')
results


Decision Tree test RMSE: 0.6020
Linear Regression test RMSE: 0.5092
LASSO test RMSE: 0.5112


EBM test RMSE: 0.4914


NAM test RMSE: 0.5480


,model,test rmse,params
0,Decision Tree,0.601973,
1,Linear Regression,0.509171,
2,LASSO,0.511179,
3,EBM,0.491406,
4,NAM,0.547955,


In [7]:
# Parameter counts, as reported in the paper (Table II)
def count_params(name, model):
    if name in ('Linear Regression', 'LASSO'):
        return model.coef_.size + 1
    if name == 'EBM':
        return sum(scores.size for scores in model.term_scores_)
    if name == 'NAM':
        # dnamite >= 1.0: ensemble of nn.Modules (paper used an earlier build, 11,361 params)
        return sum(sum(p.numel() for p in mod.parameters()) for mod in model.models)
    return 'n/a'

for idx, row in results.iterrows():
    results.loc[idx, 'params'] = count_params(row['model'], models[row['model']][0])
results


,model,test rmse,params
0,Decision Tree,0.601973,n/a
1,Linear Regression,0.509171,6
2,LASSO,0.511179,6
3,EBM,0.491406,7875
4,NAM,0.547955,225750


In [8]:
results.to_csv(f'experiments/results_interpretable_{DATASET}.csv', index=False)


**Expected results** (paper, Table II, test RMSE):

| Model | LIVE-Netflix-II | Waterloo 3 |
|---|---|---|
| Decision Tree (d=5) | 0.6020 | 0.5909 |
| Linear Regression | 0.5092 | 0.6695 |
| LASSO | 0.5461 | 0.6587 |
| EBM | 0.4914 | 0.4722 |
| NAM | 0.5480 | 0.5748 |
| **TSKAN** (notebooks 01/02) | **0.2169** | **0.4716** |

Notes:

- **LASSO**: the paper's reported values (0.5461 / 0.6587) were computed on the
  *validation* set (a bug in the original notebooks); here the test set is used,
  as in the other rows.
- **NAM**: the parameter count depends on the `dnamite` build; `dnamite>=1.0`
  trains an ensemble of five models (paper used an earlier single-model build).
